# 법인카드 이상거래 탐지 — ECOD / COPOD 비교 실험

**목적**: Isolation Forest(현재 MVP 후보) 대비 튜닝 불필요·결정론적 비지도 이상탐지 모델인 **ECOD**·**COPOD**를
동일 데이터·동일 피처셋·동일 평가 방식으로 비교한다.

**전제 조건**
- 이 노트북은 `법인카드_이상거래_전처리_v3_세그먼트플래그.ipynb`가 생성한 산출물을 그대로 읽는다:
  `../ml/.data/processed/train_processed.csv`, `test_processed.csv`, `feature_tiers.json`
- 피처셋·전처리 로직은 `법인카드_이상거래_모델링_v2_최종test평가.ipynb`(최종 IF 평가)와 **완전히 동일**하게
  맞췄다 — Tier0(14개) + `일시불할부구분코드`, `거래일자`·`거래연월` 제외, NaN은 train median 대체,
  원-핫 인코딩 후 24컬럼. 이 노트북만 따로 실행해도 IF 결과와 동일 조건에서 비교 가능하다.
- `pyod` 패키지 필요: `pip install pyod`

**핵심 차이점 (IF 대비)**
| | Isolation Forest | ECOD | COPOD |
|---|---|---|---|
| 방식 | 무작위 분기 기반 고립 | 피처별 경험적 누적분포(CDF) 꼬리 확률 | 피처별 코퓰라 기반 꼬리 확률 |
| 하이퍼파라미터 | n_estimators, max_samples 등 튜닝 필요 | **없음** | **없음** |
| 재현성 | random_state에 따라 미세하게 다름 | **완전 결정론적** | **완전 결정론적** |
| 저정보 원-핫 컬럼 취약성 | 있음(무작위 피처 선택 희석, §3 확인됨) | 낮음(전 피처를 결정론적으로 반영) | 낮음(전 피처를 결정론적으로 반영) |
| 점수 성격 | 0~1, 비보정 | outlier score(합산 꼬리 확률, 비음수) | outlier score(코퓰라 기반, 비음수) |


## 1. 데이터 로드 및 최종 피처 행렬 구성 (IF 노트북과 동일 로직)

In [1]:
import json
import os

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

pd.set_option('display.max_columns', 60)

DATA_DIR = 'data-2/processed'
TRAIN_PATH = os.path.join(DATA_DIR, 'train_processed.csv')
TEST_PATH = os.path.join(DATA_DIR, 'test_processed.csv')
TIERS_PATH = os.path.join(DATA_DIR, 'feature_tiers.json')

train_df = pd.read_csv(TRAIN_PATH, low_memory=False)
test_df = pd.read_csv(TEST_PATH, low_memory=False)
with open(TIERS_PATH, encoding='utf-8') as f:
    FEATURE_TIERS = json.load(f)

print(f"train_df: {train_df.shape}")
print(f"test_df : {test_df.shape}")
print(f"test_df에 카드_train노출여부 존재: {'카드_train노출여부' in test_df.columns}")


train_df: (1482969, 49)
test_df : (469902, 48)
test_df에 카드_train노출여부 존재: False


In [2]:
# IF 최종 노트북(모델링_v2)과 완전히 동일한 피처 구성 로직
NAN_FILL_COLS = ['사용자평균사용액_확장', '사용자표준편차_확장', '거래금액_Zscore_확장']
ONEHOT_COLS = ['거래요일_한글', '시간대구간', '일시불할부구분코드']
DROP_FROM_MODEL = ['거래일자', '거래연월']

# feature_tiers.json이 구버전(가맹점 피처 미제외)이라, IF 최종본과 동일한 Tier0(14개)를 직접 고정
TIER0_CORRECTED = [
    '승인시간대', '통합승인금액', '거래일자', '거래연월', '거래요일_한글', '시간대구간',
    '월말여부', '취소성거래_추정', '최근7일사용횟수', '카드누적사용액',
    '사용자평균사용액_확장', '사용자표준편차_확장', '거래금액_Zscore_확장', '카드첫거래여부',
]
assert len(TIER0_CORRECTED) == 14

FINAL_FEATURE_COLS = TIER0_CORRECTED + ['일시불할부구분코드']
print(f"최종 피처 목록 ({len(FINAL_FEATURE_COLS)}개):")
print(FINAL_FEATURE_COLS)

# median은 반드시 train 기준으로만 계산 — test 정보가 전처리에 섞이지 않도록
_fill_values = train_df[NAN_FILL_COLS].median()


def build_model_matrix(df, fill_values=_fill_values):
    cols = [c for c in FINAL_FEATURE_COLS if c not in DROP_FROM_MODEL]
    X = df[cols].copy()
    X[NAN_FILL_COLS] = X[NAN_FILL_COLS].fillna(fill_values)
    X['월말여부'] = X['월말여부'].astype(int)
    X = pd.get_dummies(X, columns=ONEHOT_COLS, drop_first=False)

    remaining_na = X.isna().sum()
    remaining_na = remaining_na[remaining_na > 0]
    if len(remaining_na) > 0:
        print("  [경고] 처리 안 된 NaN 발견 — 0으로 채우고 진행:")
        print(remaining_na)
        X = X.fillna(0)
    return X


X_train = build_model_matrix(train_df)
X_test = build_model_matrix(test_df)

# train/test 컬럼이 원-핫 인코딩 후 서로 다를 수 있으므로(예: 특정 카테고리가 한쪽에만 존재) 맞춰준다
X_train, X_test = X_train.align(X_test, join='outer', axis=1, fill_value=0)

y_test = test_df['이상거래여부'].values

print(f"\nX_train shape: {X_train.shape}")   # 24컬럼이어야 함 (IF 최종본과 동일)
print(f"X_test  shape: {X_test.shape}")
assert X_train.isna().sum().sum() == 0
assert X_test.isna().sum().sum() == 0
print(f"test 이상거래 비율: {y_test.mean():.4%}")

최종 피처 목록 (15개):
['승인시간대', '통합승인금액', '거래일자', '거래연월', '거래요일_한글', '시간대구간', '월말여부', '취소성거래_추정', '최근7일사용횟수', '카드누적사용액', '사용자평균사용액_확장', '사용자표준편차_확장', '거래금액_Zscore_확장', '카드첫거래여부', '일시불할부구분코드']

X_train shape: (1482969, 23)
X_test  shape: (469902, 23)
test 이상거래 비율: 3.4888%


## 2. 공통 평가 함수 (IF 노트북 Part 3과 동일 정의)

`recall@topK% / precision@topK%`, PR-AUC를 동일하게 계산한다 — IF 결과(§4)와 숫자를 직접 비교하기 위해
지표 정의를 절대 바꾸지 않는다.

In [3]:
TOP_K_FRACTIONS = [0.01, 0.03, 0.05, 0.10]


def compute_metrics(y_true, scores, fracs=TOP_K_FRACTIONS):
    '''scores: 높을수록 이상치. IF 노트북 compute_metrics와 동일 정의.'''
    row = {'n': len(y_true), 'n_positive': int(y_true.sum())}
    row['pr_auc'] = average_precision_score(y_true, scores) if y_true.sum() > 0 else np.nan
    order = np.argsort(-scores)
    n_pos = y_true.sum()
    for frac in fracs:
        k = max(1, int(len(y_true) * frac))
        top_k_idx = order[:k]
        row[f'recall@top{int(frac*100)}%'] = y_true[top_k_idx].sum() / n_pos if n_pos > 0 else np.nan
        row[f'precision@top{int(frac*100)}%'] = y_true[top_k_idx].sum() / k
    return row


## 3. ECOD 학습 및 평가

하이퍼파라미터가 없어 튜닝 단계 없이 바로 학습·채점한다. `train` 전체로 fit 후 `test`를 **딱 한 번** 채점 —
IF와 동일한 "test는 한 번만 본다" 원칙을 그대로 지킨다.

In [10]:
from pyod.models.ecod import ECOD

X_train_arr = X_train.astype(np.float64).to_numpy()
X_test_arr = X_test.astype(np.float64).to_numpy()

ecod = ECOD(n_jobs=1)  # 우선 병렬 끄고 안정성 먼저 확보
ecod.fit(X_train_arr)
ecod_test_score = ecod.decision_function(X_test_arr)

ecod_metrics = compute_metrics(y_test, ecod_test_score)
print("=== ECOD test 성능 ===")
for k, v in ecod_metrics.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")


=== ECOD test 성능 ===
  n: 469902
  n_positive: 16394
  pr_auc: 0.4294
  recall@top1%: 0.2056
  precision@top1%: 0.7174
  recall@top3%: 0.3995
  precision@top3%: 0.4646
  recall@top5%: 0.5195
  precision@top5%: 0.3625
  recall@top10%: 0.6935
  precision@top10%: 0.2419


## 4. COPOD 학습 및 평가

In [5]:
from pyod.models.copod import COPOD

from pyod.models.copod import COPOD

copod = COPOD(n_jobs=1)
copod.fit(X_train_arr)
copod_test_score = copod.decision_function(X_test_arr)


copod_metrics = compute_metrics(y_test, copod_test_score)
print("=== COPOD test 성능 ===")
for k, v in copod_metrics.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")


=== COPOD test 성능 ===
  n: 469902
  n_positive: 16394
  pr_auc: 0.5330
  recall@top1%: 0.2377
  precision@top1%: 0.8293
  recall@top3%: 0.4804
  precision@top3%: 0.5586
  recall@top5%: 0.6111
  precision@top5%: 0.4264
  recall@top10%: 0.7653
  precision@top10%: 0.2670


## 5. Isolation Forest와 3-way 비교

IF 수치는 `isolation_forest_modeling_결과.md` / `법인카드_이상거래_모델링_v2_최종test평가.ipynb`의
최종 test 결과를 그대로 인용한다(재실행하지 않음 — 이미 확정된 test 1회 채점 원칙 유지).

In [12]:
# 참고용 고정값 — IF 최종 test 결과 (재실행 아님, 문서 인용)
IF_REFERENCE = {
    'n': 469902, 'n_positive': 16394,
    'pr_auc': 0.5865,
    'recall@top1%': 0.249, 'precision@top1%': 0.868,
    'recall@top3%': 0.526, 'precision@top3%': 0.611,
    'recall@top5%': 0.664, 'precision@top5%': 0.463,
    'recall@top10%': 0.790, 'precision@top10%': 0.276,
}

comparison = pd.DataFrame({
    'Isolation Forest(참고)': IF_REFERENCE,
    'ECOD': ecod_metrics,
    'COPOD': copod_metrics,
}).T

comparison_display = comparison.drop(columns=['n', 'n_positive'])
print(f"test 건수: {int(comparison['n'].iloc[1])}, 이상거래: {int(comparison['n_positive'].iloc[1])}")
comparison_display.round(4)


test 건수: 469902, 이상거래: 16394


,pr_auc,recall@top1%,precision@top1%,recall@top3%,precision@top3%,recall@top5%,precision@top5%,recall@top10%,precision@top10%
Isolation Forest(참고),0.5865,0.2490,0.8680,0.5260,0.6110,0.6640,0.4630,0.7900,0.2760
ECOD,0.4294,0.2056,0.7174,0.3995,0.4646,0.5195,0.3625,0.6935,0.2419
COPOD,0.5330,0.2377,0.8293,0.4804,0.5586,0.6111,0.4264,0.7653,0.2670


## 6. 점수 구간별 실측 이상거래 비율 — ECOD/COPOD 버전 (IF §7과 동일 형식)

IF에서 만들었던 10분위 보정 테이블을 ECOD/COPOD 각각에 대해서도 만든다. 두 모델을 채택 후보로 검토할 경우
이 테이블 없이는 raw score를 UI/RAG에 노출할 수 없다(IF와 동일한 이유 — 비보정 점수).

In [11]:
def decile_calibration_table(y_true, scores):
    edges = np.percentile(scores, np.arange(0, 101, 10))
    decile_idx = np.clip(np.searchsorted(edges, scores, side='right') - 1, 0, 9)
    calib = (
        pd.DataFrame({'구간': decile_idx, 'y': y_true})
        .groupby('구간')['y']
        .agg(건수='size', 실측이상거래비율='mean')
    )
    calib['실측이상거래비율(%)'] = (calib['실측이상거래비율'] * 100).round(2)
    calib['기저율대비_lift'] = (calib['실측이상거래비율'] / y_true.mean()).round(2)
    calib.index = [f'{i*10}~{(i+1)*10}%' for i in range(10)]
    return calib[['건수', '실측이상거래비율(%)', '기저율대비_lift']]


print(f"전체 기저율: {y_test.mean()*100:.2f}%\n")
print("=== ECOD 보정 테이블 ===")
display(decile_calibration_table(y_test, ecod_test_score))
print("\n=== COPOD 보정 테이블 ===")
display(decile_calibration_table(y_test, copod_test_score))


전체 기저율: 3.49%

=== ECOD 보정 테이블 ===


,건수,실측이상거래비율(%),기저율대비_lift
0~10%,46991,0.04,0.01
10~20%,46990,0.13,0.04
20~30%,46990,0.33,0.10
30~40%,46990,0.39,0.11
40~50%,46990,0.54,0.16
50~60%,46990,0.80,0.23
60~70%,46990,1.20,0.35
70~80%,46990,2.21,0.63
80~90%,46990,5.04,1.44
90~100%,46991,24.19,6.93



=== COPOD 보정 테이블 ===


,건수,실측이상거래비율(%),기저율대비_lift
0~10%,46991,0.02,0.01
10~20%,46990,0.07,0.02
20~30%,46990,0.20,0.06
30~40%,46990,0.33,0.10
40~50%,46990,0.42,0.12
50~60%,46990,0.55,0.16
60~70%,46990,0.92,0.26
70~80%,46990,1.59,0.46
80~90%,46990,4.09,1.17
90~100%,46991,26.70,7.65


## 7. 세그먼트 분석 — 재사용 카드 vs 신규 카드

IF에서 "신규 카드(콜드스타트)에서 오히려 recall이 높다"(57.7% vs 52.6%)는 결과가 나왔다. ECOD/COPOD도
같은 경향인지, 아니면 카드 이력 의존도가 다르게 나타나는지 확인한다.

In [9]:
segment_rows = {}
for seg_value, seg_label in [(True, '재사용 카드'), (False, '신규 카드')]:
    mask = (test_df['카드_train노출여부'] == seg_value).values
    y_seg = y_test[mask]

    segment_rows[('ECOD', seg_label)] = compute_metrics(y_seg, ecod_test_score[mask])
    segment_rows[('COPOD', seg_label)] = compute_metrics(y_seg, copod_test_score[mask])

segment_summary = pd.DataFrame(segment_rows).T
segment_summary.drop(columns=['n', 'n_positive']).round(4)


KeyError: '카드_train노출여부'

## 8. 결과 요약 (실행 후 직접 채워서 결과보고서에 반영)

아래는 채점 후 직접 정리해야 할 항목이다. 실행 결과를 보고 다음을 판단한다.

- **PR-AUC/recall이 IF 대비 유의미하게 높은가, 비슷한가, 낮은가?**
  - 유의미하게 높다면: ECOD/COPOD를 MVP 후보로 교체 검토 — 튜닝 불필요·결정론적이라는 부가 이점도 있음
  - 비슷하다면: IF를 유지하되, "저정보 원-핫 컬럼에 더 강건하다"는 이론적 장점이 실증되지 않았다는 뜻이므로
    굳이 교체할 실익 없음(운영 컴포넌트를 늘리는 비용 대비)
  - 낮다면: IF 유지, ECOD/COPOD는 폐기
- **세그먼트(신규 카드) 결과가 IF와 같은 방향(콜드스타트에서 더 강함)인지** — 다르다면 어느 모델이 왜
  그런지는 별도로 살펴봐야 함(피처 특성상 IF만의 우연일 수도 있음)
- **주의**: 이 비교는 "비지도 알고리즘 중 어느 게 나은가"만 답한다. 지도학습(XGBoost 등, README 기준
  top3% recall 74.7%)과의 격차는 이 실험으로 전혀 해소되지 않는다 — 그 격차는 라벨 타당성 검증이 별도로
  필요한, 완전히 다른 트랙이다.


In [15]:
from sklearn.preprocessing import StandardScaler
from pyod.models.inne import INNE
from pyod.models.loda import LODA
from pyod.models.gmm import GMM

# 스케일링은 한 번만 — fit은 train에만, test는 transform만 (median 처리와 같은 원칙)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_arr)
X_test_scaled = scaler.transform(X_test_arr)

# INNE
inne = INNE(n_estimators=200, max_samples='auto', random_state=42)
inne.fit(X_train_scaled)
inne_test_score = inne.decision_function(X_test_scaled)
print("=== INNE (스케일링 후) ===")
for k, v in compute_metrics(y_test, inne_test_score).items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

# LODA
loda = LODA()
loda.fit(X_train_scaled)
loda_test_score = loda.decision_function(X_test_scaled)
print("\n=== LODA (스케일링 후) ===")
for k, v in compute_metrics(y_test, loda_test_score).items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

# GMM — n_components 기본값(1)은 사실상 단일 가우시안이라 의미 없음, 명시 지정 필요
gmm = GMM(n_components=5, random_state=42)
gmm.fit(X_train_scaled)
gmm_test_score = gmm.decision_function(X_test_scaled)
print("\n=== GMM (스케일링 후, n_components=5) ===")
for k, v in compute_metrics(y_test, gmm_test_score).items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

=== INNE (스케일링 후) ===
  n: 469902
  n_positive: 16394
  pr_auc: 0.3762
  recall@top1%: 0.1233
  precision@top1%: 0.4301
  recall@top3%: 0.3928
  precision@top3%: 0.4568
  recall@top5%: 0.6665
  precision@top5%: 0.4651
  recall@top10%: 0.7977
  precision@top10%: 0.2783

=== LODA (스케일링 후) ===
  n: 469902
  n_positive: 16394
  pr_auc: 0.0283
  recall@top1%: 0.0095
  precision@top1%: 0.0332
  recall@top3%: 0.0164
  precision@top3%: 0.0191
  recall@top5%: 0.0218
  precision@top5%: 0.0152
  recall@top10%: 0.0466
  precision@top10%: 0.0163

=== GMM (스케일링 후, n_components=5) ===
  n: 469902
  n_positive: 16394
  pr_auc: 0.3418
  recall@top1%: 0.0941
  precision@top1%: 0.3282
  recall@top3%: 0.4302
  precision@top3%: 0.5003
  recall@top5%: 0.6861
  precision@top5%: 0.4787
  recall@top10%: 0.7148
  precision@top10%: 0.2494


## 1. Fold 비교 코드 — IF vs INNE vs GMM(vs COPOD)


In [26]:
fold_results = []  # 반드시 이 줄부터 다시 실행 — 예전 값 완전히 비우기

for fold_idx, (tr_idx, va_idx) in enumerate(sgkf.split(train_df, y_all, groups)):
    fold_train_df = train_df.iloc[tr_idx]
    fold_valid_df = train_df.iloc[va_idx]

    fold_fill = fold_train_df[NAN_FILL_COLS].median()
    X_ftr = build_model_matrix(fold_train_df, fill_values=fold_fill)
    X_fva = build_model_matrix(fold_valid_df, fill_values=fold_fill)
    X_ftr, X_fva = X_ftr.align(X_fva, join='outer', axis=1, fill_value=0)

    y_fva = fold_valid_df['이상거래여부'].values
    Xtr_arr = X_ftr.astype(np.float64).to_numpy()
    Xva_arr = X_fva.astype(np.float64).to_numpy()

    scaler = StandardScaler().fit(Xtr_arr)
    Xtr_scaled = scaler.transform(Xtr_arr)
    Xva_scaled = scaler.transform(Xva_arr)

    iforest = IsolationForest(n_estimators=200, max_samples='auto', contamination='auto',
                               random_state=42, n_jobs=-1)
    iforest.fit(Xtr_arr)
    if_score = -iforest.decision_function(Xva_arr)

    copod = COPOD(n_jobs=1)
    copod.fit(Xtr_arr)
    copod_score = copod.decision_function(Xva_arr)

    inne = INNE(n_estimators=200, max_samples='auto', random_state=42)
    inne.fit(Xtr_scaled)
    inne_score = inne.decision_function(Xva_scaled)

    gmm = GMM(n_components=5, random_state=42)
    gmm.fit(Xtr_scaled)
    gmm_score = gmm.decision_function(Xva_scaled)

    for name, score in [('IF', if_score), ('COPOD', copod_score),
                         ('INNE', inne_score), ('GMM', gmm_score)]:
        m = compute_metrics(y_fva, score)
        m['fold'] = fold_idx
        m['model'] = name
        fold_results.append(m)

    print(f"fold {fold_idx} 완료 (valid n={len(y_fva)}, 이상거래={int(y_fva.sum())})")

fold_df = pd.DataFrame(fold_results)
print(fold_df.shape)  # 반드시 (20, ...)이어야 함
print(fold_df.pivot(index='fold', columns='model', values='pr_auc'))

fold 0 완료 (valid n=296613, 이상거래=11135)
fold 1 완료 (valid n=296522, 이상거래=11090)
fold 2 완료 (valid n=296582, 이상거래=11117)
fold 3 완료 (valid n=296681, 이상거래=11163)
fold 4 완료 (valid n=296571, 이상거래=11109)
(20, 13)
model     COPOD       GMM        IF      INNE
fold                                         
0      0.550191  0.328544  0.564083  0.417320
1      0.602909  0.375784  0.616367  0.493818
2      0.605173  0.290843  0.623083  0.474622
3      0.599820  0.345556  0.626990  0.525990
4      0.477989  0.296867  0.514423  0.352780


In [27]:
from scipy import stats

if_scores = fold_df[fold_df['model'] == 'IF'].sort_values('fold')['pr_auc'].values
copod_scores = fold_df[fold_df['model'] == 'COPOD'].sort_values('fold')['pr_auc'].values

diff = if_scores - copod_scores
print("fold별 차이:", diff)
print("평균 차이:", diff.mean(), "표준편차:", diff.std(ddof=1))

t_stat, p_value = stats.ttest_rel(if_scores, copod_scores)
print(f"t={t_stat:.3f}, p={p_value:.4f}")

fold별 차이: [0.01389248 0.01345885 0.01790983 0.0271706  0.03643474]
평균 차이: 0.021773298919059082 표준편차: 0.00987740682091728
t=4.929, p=0.0079


In [19]:
from pyod.models.cblof import CBLOF
from pyod.models.pca import PCA

# CBLOF — 클러스터 분리가 안 되면 에러날 수 있어 n_clusters를 조정하며 시도
cblof = None
for n_clusters in [8, 10, 15]:
    try:
        cblof = CBLOF(n_clusters=n_clusters, random_state=42)
        cblof.fit(X_train_scaled)
        print(f"CBLOF n_clusters={n_clusters} 성공")
        break
    except ValueError as e:
        print(f"CBLOF n_clusters={n_clusters} 실패: {e}")

if cblof is not None:
    cblof_test_score = cblof.decision_function(X_test_scaled)
    print("=== CBLOF ===")
    for k, v in compute_metrics(y_test, cblof_test_score).items():
        print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

# PCA — 재구성 오차 기반, 스케일링 필요
pca = PCA(random_state=42, weighted=False)
pca.fit(X_train_scaled)
pca_test_score = pca.decision_function(X_test_scaled)
print("=== PCA (weighted=False) ===")
for k, v in compute_metrics(y_test, pca_test_score).items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

CBLOF n_clusters=8 성공
=== CBLOF ===
  n: 469902
  n_positive: 16394
  pr_auc: 0.3858
  recall@top1%: 0.1576
  precision@top1%: 0.5499
  recall@top3%: 0.4423
  precision@top3%: 0.5144
  recall@top5%: 0.5661
  precision@top5%: 0.3950
  recall@top10%: 0.7739
  precision@top10%: 0.2700
=== PCA (weighted=False) ===
  n: 469902
  n_positive: 16394
  pr_auc: 0.3833
  recall@top1%: 0.1571
  precision@top1%: 0.5480
  recall@top3%: 0.4386
  precision@top3%: 0.5101
  recall@top5%: 0.5620
  precision@top5%: 0.3921
  recall@top10%: 0.7702
  precision@top10%: 0.2687
